# Datos reales — engranajes XYZ etiquetados

Carga de datos de medición reales: nube de puntos XYZ en formato `.raw` y máscara de segmentación de defecto en formato YOLO (`.txt`).

In [50]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import os
import sys

# Añadir el directorio raíz del proyecto al path para importar camera.raw
sys.path.insert(0, os.path.abspath(".."))
import camera.raw as raw

## 1. Cargar datos XYZ

El archivo `.raw` tiene header `IMG_INFO` + nr/nc (int32 LE) + tag `F34B` (3 canales float32 → X, Y, Z).

In [51]:
SAMPLE = "151225_160136_P44512_S2_7761"
BASE = f"../gears_defectos_xyz_etiquetados/{SAMPLE}/debug/entities"

xyz_path  = f"{BASE}/save_xyz/{SAMPLE}_save_xyz.raw"
mask_path = f"{BASE}/rgb_seg/{SAMPLE}_rgb_seg.txt"

# Leer XYZ: shape (H, W, 3) → canales [X, Y, Z]
xyz = raw.read_img_raw(xyz_path)
H, W = xyz.shape[:2]
X, Y, Z = xyz[:, :, 0], xyz[:, :, 1], xyz[:, :, 2]

print(f"Shape XYZ : {xyz.shape}")
print(f"X range   : [{X.min():.3f}, {X.max():.3f}] mm")
print(f"Y range   : [{Y.min():.3f}, {Y.max():.3f}] mm")
print(f"Z range   : [{Z.min():.3f}, {Z.max():.3f}] mm")

Shape XYZ : (790, 1129, 3)
X range   : [-59.985, 88.030] mm
Y range   : [-48.351, 48.657] mm
Z range   : [0.000, 21.330] mm


## 2. Cargar máscara de segmentación

Formato YOLO segmentación: una línea por objeto → `clase x1 y1 x2 y2 ... xN yN` con coordenadas normalizadas a [0,1].

In [52]:
def load_yolo_seg_mask(txt_path, img_h, img_w):
    """
    Lee una máscara YOLO segmentation (polígono normalizado) y devuelve:
      - polygons : lista de arrays (N,2) en píxeles [col, row]
      - mask     : imagen binaria (img_h, img_w) uint8
    """
    polygons = []
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    with open(txt_path, "r") as f:
        for line in f:
            vals = line.strip().split()
            if len(vals) < 5:
                continue
            # primer valor = clase, el resto son pares x,y normalizados
            coords = np.array(vals[1:], dtype=float).reshape(-1, 2)
            pts_px = (coords * np.array([img_w, img_h])).astype(np.int32)
            polygons.append(pts_px)
            cv2.fillPoly(mask, [pts_px], 1)
    return polygons, mask


polygons, defect_mask = load_yolo_seg_mask(mask_path, H, W)

print(f"Número de polígonos : {len(polygons)}")
for i, p in enumerate(polygons):
    print(f"  Polígono {i}: {len(p)} vértices")
print(f"Píxeles en máscara  : {defect_mask.sum()}")

Número de polígonos : 1
  Polígono 0: 93 vértices
Píxeles en máscara  : 5489


## 3. Visualización

In [53]:
plt.close("all")
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# — Mapa Z completo
im0 = axes[0].imshow(Z, cmap="turbo", aspect="auto")
fig.colorbar(im0, ax=axes[0], label="Z (mm)")
axes[0].set_title("Mapa Z")

# — Mapa Z con contorno de la máscara superpuesto
axes[1].imshow(Z, cmap="turbo", aspect="auto")
for pts in polygons:
    closed = np.vstack([pts, pts[0]])
    axes[1].plot(closed[:, 0], closed[:, 1], "w-", linewidth=1.5)
axes[1].set_title("Z + contorno defecto")

# — Zoom en la región del defecto (bounding box del polígono)
if polygons:
    all_pts = np.vstack(polygons)
    r0, r1 = all_pts[:, 1].min(), all_pts[:, 1].max()
    c0, c1 = all_pts[:, 0].min(), all_pts[:, 0].max()
    pad = 20
    r0, r1 = max(0, r0 - pad), min(H, r1 + pad)
    c0, c1 = max(0, c0 - pad), min(W, c1 + pad)
    Z_crop = Z[r0:r1, c0:c1]
    mask_crop = defect_mask[r0:r1, c0:c1]
    im2 = axes[2].imshow(Z_crop, cmap="turbo", aspect="auto")
    fig.colorbar(im2, ax=axes[2], label="Z (mm)")
    # overlay semitransparente de la máscara
    overlay = np.zeros((*Z_crop.shape, 4), dtype=float)
    overlay[mask_crop == 1] = [1, 1, 0, 0.35]   # amarillo semitransparente
    axes[2].imshow(overlay, aspect="auto")
    axes[2].set_title(f"Zoom defecto ({r1-r0}×{c1-c0} px)")

plt.suptitle(SAMPLE, fontsize=10)
plt.tight_layout()
plt.show()

invalid command name "137241378545088process_stream_events"
    while executing
"137241378545088process_stream_events"
    ("after" script)


## 4. Normalización polinomial local

La pieza es un eje de forja (geometría cilíndrica compleja, sin modelo analítico previo).  
Estrategia: ajustar un **polinomio 2D de grado `POLY_DEG`** a los píxeles de una **corona alrededor del defecto** (excluyendo los píxeles del defecto mismo). Eso da la superficie nominal `Z_nom`. Entonces:

$$Z_{\text{diff}} = Z - Z_{\text{nom}}$$

La corona se construye dilatando la máscara `RING_PX` píxeles y restando la máscara original.

In [54]:

# ── Parámetros ──────────────────────────────────────────────────────────────
RING_LOCAL_MM = 8.0   # anchura del anillo de muestras alrededor del defecto (mm)
POLY_DEG      = 2     # grado del polinomio tapa (solo para referencia RMS)
K_FACETS      = 2     # número de facetas (tapa + pared cilíndrica)
SG_WIN_PX     = 31    # ventana SG 2D en píxeles (impar) para estimar gradiente
# ────────────────────────────────────────────────────────────────────────────

from scipy.interpolate import griddata

# Píxeles válidos (Z=0 → sin medición)
valid_mask = np.isfinite(Z) & (Z != 0)

# Resolución aproximada (mm/px)
x_valid = X[valid_mask];  y_valid = Y[valid_mask]
dx = (x_valid.max() - x_valid.min()) / W
dy = (y_valid.max() - y_valid.min()) / H
pix_mm = (dx + dy) / 2
ring_local_px = int(round(RING_LOCAL_MM / pix_mm)) | 1
print(f"Resolución media: {pix_mm:.4f} mm/px  →  anillo {RING_LOCAL_MM} mm = {ring_local_px} px")

# ── SG 2D (paraboloide): devuelve tensor (H,W,6) con coefs [a,b,c,d,e,f0] ──
def sg2d_jacobian_projection_cv2(Z, window_size=11):
    assert window_size % 2 == 1
    hw = window_size // 2
    y_idx, x_idx = np.mgrid[-hw:hw+1, -hw:hw+1]
    regressors = np.stack([
        x_idx**2, y_idx**2, x_idx * y_idx,
        x_idx, y_idx, np.ones_like(x_idx)
    ], axis=-1)
    A_mat = regressors.reshape(-1, 6)
    pseudo = np.linalg.inv(A_mat.T @ A_mat) @ A_mat.T
    kernels = pseudo.reshape(6, window_size, window_size)
    return np.stack(
        [cv2.filter2D(Z.astype(np.float64), -1, k,
                      borderType=cv2.BORDER_REFLECT) for k in kernels],
        axis=-1
    )

# ── Gradiente SG 2D en espacio píxel ────────────────────────────────────────
print(f"Ventana SG 2D: {SG_WIN_PX} px")
Z_filled  = np.where(valid_mask, Z, 0.0)
coeffs_px = sg2d_jacobian_projection_cv2(Z_filled, SG_WIN_PX)
gx_px = coeffs_px[..., 3]
gy_px = coeffs_px[..., 4]

# ── Anillo de muestras ───────────────────────────────────────────────────────
kernel_rl     = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (ring_local_px, ring_local_px))
mask_outer_rl = cv2.dilate(defect_mask, kernel_rl).astype(bool)
mask_ring_rl  = mask_outer_rl & ~defect_mask.astype(bool) & valid_mask
print(f"Anillo: {RING_LOCAL_MM} mm → {ring_local_px} px  |  puntos={mask_ring_rl.sum()}")

# ── Función de diseño polinomial 2D ─────────────────────────────────────────
def poly2d_design(x, y, deg=2):
    terms = [np.ones_like(x)]
    for d in range(1, deg + 1):
        for k in range(d + 1):
            terms.append(x ** (d - k) * y ** k)
    return np.column_stack(terms)

# ── Separación tapa / pared por magnitud del gradiente SG (Otsu) ─────────────
r_idx, c_idx = np.where(mask_ring_rl)
xp  = X[r_idx, c_idx].astype(float)
yp  = Y[r_idx, c_idx].astype(float)
zp  = Z[r_idx, c_idx].astype(float)
gxp = gx_px[r_idx, c_idx]
gyp = gy_px[r_idx, c_idx]

grad_mag = np.sqrt(gxp**2 + gyp**2)
grad_u8  = (np.clip(grad_mag / (grad_mag.max() + 1e-12), 0, 1) * 255).astype(np.uint8)
thr_otsu, _ = cv2.threshold(grad_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
thr_mag  = (thr_otsu / 255.0) * (grad_mag.max() + 1e-12)

sel_tapa  = grad_mag <= thr_mag
sel_pared = ~sel_tapa
labels    = sel_pared.astype(np.int16)

print(f"\nGeometría cilíndrica — 2 facetas:")
print(f"  Faceta 0 (tapa ): {sel_tapa.sum():5d} px  |  |∇| medio={grad_mag[sel_tapa].mean():.4f}")
print(f"  Faceta 1 (pared): {sel_pared.sum():5d} px  |  |∇| medio={grad_mag[sel_pared].mean():.4f}")

# Para visualización mantenemos coefs polinomiales
coeffs_list = []
for sel in [sel_tapa, sel_pared]:
    A = poly2d_design(xp[sel], yp[sel], POLY_DEG)
    coef, _, _, _ = np.linalg.lstsq(A, zp[sel], rcond=None)
    coeffs_list.append(coef)

# ── Frontera tapa/pared ───────────────────────────────────────────────────────
mask_tapa_img  = np.zeros((H, W), dtype=np.uint8)
mask_pared_img = np.zeros((H, W), dtype=np.uint8)
mask_tapa_img [r_idx[sel_tapa],  c_idx[sel_tapa]]  = 1
mask_pared_img[r_idx[sel_pared], c_idx[sel_pared]] = 1

k3 = np.ones((3, 3), dtype=np.uint8)
boundary_img = (
    (cv2.dilate(mask_tapa_img,  k3).astype(bool) & mask_pared_img.astype(bool)) |
    (cv2.dilate(mask_pared_img, k3).astype(bool) & mask_tapa_img.astype(bool))
)
br_idx, bc_idx = np.where(boundary_img)
xb = X[br_idx, bc_idx].astype(float)
yb = Y[br_idx, bc_idx].astype(float)
print(f"\n  Puntos de frontera tapa/pared: {len(xb)}")

# ── RANSAC: ajuste polinomial de la frontera en espacio píxel ────────────────
if len(xb) >= 3:
    xb_px = bc_idx.astype(float)
    yb_px = br_idx.astype(float)
    if np.std(xb_px) >= np.std(yb_px):
        t_pts, s_pts = yb_px, xb_px   # col = f(row)
        mode = 'col_of_row'
    else:
        t_pts, s_pts = xb_px, yb_px   # row = f(col)
        mode = 'row_of_col'
    POLY_RANSAC = 2; N_ITER_R = 300; THR_R_PX = 3.0
    rng_r = np.random.default_rng(42)
    best_inliers = np.zeros(len(t_pts), dtype=bool)
    best_coef_r  = np.polyfit(t_pts, s_pts, POLY_RANSAC)
    for _ in range(N_ITER_R):
        idx_s    = rng_r.choice(len(t_pts), POLY_RANSAC + 1, replace=False)
        coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
        inliers  = np.abs(np.polyval(coef_try, t_pts) - s_pts) < THR_R_PX
        if inliers.sum() > best_inliers.sum():
            best_inliers = inliers; best_coef_r = coef_try
    if best_inliers.sum() >= POLY_RANSAC + 1:
        best_coef_r = np.polyfit(t_pts[best_inliers], s_pts[best_inliers], POLY_RANSAC)
    res_boundary = float(np.mean(np.abs(np.polyval(best_coef_r, t_pts[best_inliers]) - s_pts[best_inliers])))
    print(f"  RANSAC frontera: {best_inliers.sum()}/{len(t_pts)} inliers  err={res_boundary:.2f}px")
    arista_existe = True
else:
    arista_existe = False; best_coef_r = None; mode = None
    print("  No hay suficientes puntos de frontera → faceta única")

# ── Bounding box de evaluación ───────────────────────────────────────────────
r0_bb = max(0, r_idx.min() - ring_local_px)
r1_bb = min(H, r_idx.max() + ring_local_px + 1)
c0_bb = max(0, c_idx.min() - ring_local_px)
c1_bb = min(W, c_idx.max() + ring_local_px + 1)

X_ev = X[r0_bb:r1_bb, c0_bb:c1_bb].astype(float)
Y_ev = Y[r0_bb:r1_bb, c0_bb:c1_bb].astype(float)
hr, wc = X_ev.shape

# ── Asignación tapa/pared del defecto por RANSAC ─────────────────────────────
def_in_bb = defect_mask[r0_bb:r1_bb, c0_bb:c1_bb].astype(bool)

if arista_existe:
    cols_ev_arr = np.arange(c0_bb, c1_bb, dtype=float)
    rows_ev_arr = np.arange(r0_bb, r1_bb, dtype=float)
    CC_ev, RR_ev = np.meshgrid(cols_ev_arr, rows_ev_arr)
    if mode == 'col_of_row':
        t_ev = RR_ev; s_ev = CC_ev
        t_tapa = r_idx[sel_tapa].astype(float)
        s_tapa = c_idx[sel_tapa].astype(float)
    else:
        t_ev = CC_ev; s_ev = RR_ev
        t_tapa = c_idx[sel_tapa].astype(float)
        s_tapa = r_idx[sel_tapa].astype(float)
    side_ev   = s_ev   - np.polyval(best_coef_r, t_ev)
    side_tapa = float(np.mean(s_tapa - np.polyval(best_coef_r, t_tapa)))
    is_pared  = (side_ev < 0) if (side_tapa > 0) else (side_ev > 0)
    nearest_k = is_pared.astype(int)
else:
    is_pared  = np.zeros((hr, wc), dtype=bool)
    nearest_k = np.zeros((hr, wc), dtype=int)

# ── Z_nom: interpolación griddata de la superficie de alrededor ───────────────
# Para cada faceta, los puntos fuente son los píxeles del ANILLO de esa faceta.
# Los puntos destino son los píxeles del DEFECTO de esa faceta.
# Se usa interpolación lineal → continuación suave sin saltos en la frontera.

# Puntos fuente en píxeles (col, row) → usar coordenadas imagen para interpolar
# (reflejan la geometría de la imagen mejor que X_mm, Y_mm en este scanner)
src_tapa_c  = c_idx[sel_tapa].astype(float)
src_tapa_r  = r_idx[sel_tapa].astype(float)
src_tapa_z  = zp[sel_tapa]

src_pared_c = c_idx[sel_pared].astype(float)
src_pared_r = r_idx[sel_pared].astype(float)
src_pared_z = zp[sel_pared]

# Puntos destino dentro del defecto para cada faceta
def_pared_bb = def_in_bb & is_pared
def_tapa_bb  = def_in_bb & ~is_pared

Z_nom_ev = np.zeros((hr, wc), dtype=float)

# — Tapa: interpolación lineal + fallback nearest
rows_def_tapa, cols_def_tapa = np.where(def_tapa_bb)
if len(rows_def_tapa) > 0:
    dst_c_t = (cols_def_tapa + c0_bb).astype(float)
    dst_r_t = (rows_def_tapa + r0_bb).astype(float)
    z_int_t = griddata(
        np.column_stack([src_tapa_c, src_tapa_r]),
        src_tapa_z,
        np.column_stack([dst_c_t, dst_r_t]),
        method='linear'
    )
    # Fallback nearest para píxeles fuera de la envolvente convexa
    nan_mask = np.isnan(z_int_t)
    if nan_mask.any():
        z_near_t = griddata(
            np.column_stack([src_tapa_c, src_tapa_r]),
            src_tapa_z,
            np.column_stack([dst_c_t[nan_mask], dst_r_t[nan_mask]]),
            method='nearest'
        )
        z_int_t[nan_mask] = z_near_t
    Z_nom_ev[rows_def_tapa, cols_def_tapa] = z_int_t

# — Pared: interpolación lineal + fallback nearest
rows_def_pared, cols_def_pared = np.where(def_pared_bb)
if len(rows_def_pared) > 0:
    dst_c_p = (cols_def_pared + c0_bb).astype(float)
    dst_r_p = (rows_def_pared + r0_bb).astype(float)
    z_int_p = griddata(
        np.column_stack([src_pared_c, src_pared_r]),
        src_pared_z,
        np.column_stack([dst_c_p, dst_r_p]),
        method='linear'
    )
    nan_mask = np.isnan(z_int_p)
    if nan_mask.any():
        z_near_p = griddata(
            np.column_stack([src_pared_c, src_pared_r]),
            src_pared_z,
            np.column_stack([dst_c_p[nan_mask], dst_r_p[nan_mask]]),
            method='nearest'
        )
        z_int_p[nan_mask] = z_near_p
    Z_nom_ev[rows_def_pared, cols_def_pared] = z_int_p

print(f"  Interpolación: {len(rows_def_tapa)} px tapa + {len(rows_def_pared)} px pared")

# Z_nom: solo se sustituye dentro de la máscara del defecto
Z_nom = Z.copy()
Z_nom[r0_bb:r1_bb, c0_bb:c1_bb] = np.where(def_in_bb, Z_nom_ev, Z[r0_bb:r1_bb, c0_bb:c1_bb])
Z_diff = np.where(valid_mask, Z - Z_nom, np.nan)

z_def = Z_diff[defect_mask.astype(bool)]
print(f"\nZ_diff en máscara defecto:  min={np.nanmin(z_def):.4f}  max={np.nanmax(z_def):.4f}  std={np.nanstd(z_def):.4f} mm")


Resolución media: 0.1269 mm/px  →  anillo 8.0 mm = 63 px
Ventana SG 2D: 31 px
Anillo: 8.0 mm → 63 px  |  puntos=11655

Geometría cilíndrica — 2 facetas:
  Faceta 0 (tapa ):  6164 px  |  |∇| medio=0.0077
  Faceta 1 (pared):  5491 px  |  |∇| medio=0.0889

  Puntos de frontera tapa/pared: 182
  RANSAC frontera: 158/182 inliers  err=1.36px
  Interpolación: 3360 px tapa + 2129 px pared

Z_diff en máscara defecto:  min=-0.6528  max=0.6656  std=0.2609 mm


/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
  coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
  coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
  coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
  coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
  coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
  coef_try = np.polyfit(t_pts[idx_s], s_pts[idx_s], POLY_RANSAC)
/tmp/ipykernel_25801/3582882263.py:121: RankWarning: Polyfit may be poorly conditioned
 

In [55]:

# — Bounding box para los plots (reutiliza r0_bb calculado en la celda anterior)
def crop(arr):
    return arr[r0_bb:r1_bb, c0_bb:c1_bb]

Z_cr      = crop(Z)
Z_nom_cr  = crop(Z_nom)
Z_diff_cr = crop(Z_diff)
mask_cr   = crop(defect_mask.astype(bool))
valid_cr  = crop(valid_mask)

# Rango Z válido
vmin_z = float(np.nanmin(Z_cr[valid_cr]))
vmax_z = float(np.nanmax(Z_cr[valid_cr]))

# Mapa de facetas en el crop
labels_img = np.full((H, W), -1, dtype=np.int16)
labels_img[r_idx, c_idx] = labels
labels_cr  = crop(labels_img)

# ── Arco de la arista: reutiliza best_coef_r y mode de la celda anterior ──
res_px = float(np.mean(np.abs(np.polyval(best_coef_r, t_pts[best_inliers]) - s_pts[best_inliers])))

# Curva extendida más allá de los puntos para mostrar continuidad
margin_arc = 80   # píxeles de extensión
t_arc = np.linspace(t_pts.min() - margin_arc, t_pts.max() + margin_arc, 3000)
s_arc = np.polyval(best_coef_r, t_arc)

if mode == 'col_of_row':
    arc_r_full, arc_c_full = t_arc, s_arc
else:
    arc_c_full, arc_r_full = t_arc, s_arc

arc_c_crop = arc_c_full - c0_bb
arc_r_crop = arc_r_full - r0_bb
in_arc = ((arc_c_crop >= 0) & (arc_c_crop < Z_cr.shape[1]) &
          (arc_r_crop >= 0) & (arc_r_crop < Z_cr.shape[0]))
arc_c = arc_c_crop[in_arc]
arc_r = arc_r_crop[in_arc]

# Puntos de frontera en coordenadas del crop
bc_crop = bc_idx - c0_bb
br_crop = br_idx - r0_bb
in_crop = ((br_crop >= 0) & (br_crop < Z_cr.shape[0]) &
           (bc_crop >= 0) & (bc_crop < Z_cr.shape[1]))

# — Límites colorbar Z_diff simétrico
vmax_diff = np.nanpercentile(np.abs(Z_diff_cr[mask_cr]), 99)

plt.close("all")
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1 — Z medido
im0 = axes[0].imshow(Z_cr, cmap="turbo", aspect="auto", vmin=vmin_z, vmax=vmax_z)
fig.colorbar(im0, ax=axes[0], label="mm")
axes[0].set_title("Z medido")

# Panel 2 — Z nominal
im1 = axes[1].imshow(Z_nom_cr, cmap="turbo", aspect="auto", vmin=vmin_z, vmax=vmax_z)
fig.colorbar(im1, ax=axes[1], label="mm")
axes[1].set_title("Z nominal (defecto inpaintado)")
axes[1].contour(mask_cr, levels=[0.5], colors='white', linewidths=1.2)

# Panel 3 — Z_diff + arco
im2 = axes[2].imshow(Z_diff_cr, cmap="RdBu_r", aspect="auto",
                     vmin=-vmax_diff, vmax=vmax_diff)
fig.colorbar(im2, ax=axes[2], label="mm")
axes[2].set_title("Z_diff = Z − Z_nom")
for pts in polygons:
    pts_local = pts - np.array([c0_bb, r0_bb])
    closed = np.vstack([pts_local, pts_local[0]])
    axes[2].plot(closed[:, 0], closed[:, 1], "k-", linewidth=1.2)
if arista_existe and len(arc_c) > 0:
    axes[2].scatter(arc_c, arc_r, s=2, c='cyan', alpha=0.8, linewidths=0,
                    label=f'arista RANSAC (err={res_px:.1f}px)')
    axes[2].legend(fontsize=7, loc='upper right')

# Panel 4 — Facetas + frontera + arco
cmap_f = plt.colormaps.get_cmap("Set1").resampled(K_FACETS)
facet_vis = np.zeros((*Z_cr.shape, 4), dtype=float)
for k in range(K_FACETS):
    c = cmap_f(k)
    facet_vis[labels_cr == k] = [c[0], c[1], c[2], 0.5]
mask_vis = np.zeros((*Z_cr.shape, 4), dtype=float)
mask_vis[mask_cr] = [1.0, 0.8, 0.0, 0.6]

axes[3].imshow(Z_cr, cmap="gray", aspect="auto", vmin=vmin_z, vmax=vmax_z)
axes[3].imshow(facet_vis, aspect="auto")
axes[3].imshow(mask_vis, aspect="auto")
if arista_existe:
    if in_crop.any():
        axes[3].scatter(bc_crop[in_crop], br_crop[in_crop],
                        s=12, c='magenta', alpha=1.0, linewidths=0,
                        label='frontera anillo', zorder=5)
    if len(arc_c) > 0:
        axes[3].scatter(arc_c, arc_r, s=2, c='cyan', alpha=0.8, linewidths=0,
                        label=f'arista RANSAC (err={res_px:.1f}px)', zorder=4)
    axes[3].legend(fontsize=7, loc='upper right')
    axes[3].set_title(f"Facetas + frontera (magenta) + arista (cian, err={res_px:.1f}px)")
else:
    axes[3].set_title(f"Facetas (K={K_FACETS}) + defecto")

plt.suptitle(SAMPLE, fontsize=9)
plt.tight_layout()
plt.show()


## 5. Multi-Splat: funciones

In [56]:
import time as _time
from scipy.optimize import least_squares

def gauss2d_rotated(params, x, y):
    A_p, x0_p, y0_p, sx_p, sy_p, th_p = params
    c, s = np.cos(th_p), np.sin(th_p)
    u =  (x - x0_p) * c + (y - y0_p) * s
    v = -(x - x0_p) * s + (y - y0_p) * c
    return A_p * np.exp(-0.5 * (u**2 / sx_p**2 + v**2 / sy_p**2))

def gauss_mixture(params, x, y):
    K = len(params) // 6
    out = np.zeros(len(x))
    for k in range(K):
        out += gauss2d_rotated(params[6*k:6*k+6], x, y)
    return out

def gauss_mixture_and_jac(params, x, y):
    K = len(params) // 6
    N = len(x)
    out = np.zeros(N); J = np.zeros((N, 6*K))
    for k in range(K):
        A_p, x0_p, y0_p, sx_p, sy_p, th_p = params[6*k:6*k+6]
        c, s = np.cos(th_p), np.sin(th_p)
        dxv = x - x0_p; dyv = y - y0_p
        u =  dxv*c + dyv*s; v = -dxv*s + dyv*c
        eu2 = u**2/sx_p**2; ev2 = v**2/sy_p**2
        g = A_p * np.exp(-0.5*(eu2+ev2))
        out += g
        J[:, 6*k]   = g / A_p
        J[:, 6*k+1] = g * (u*c/sx_p**2 - v*s/sy_p**2)
        J[:, 6*k+2] = g * (u*s/sx_p**2 + v*c/sy_p**2)
        J[:, 6*k+3] = g * eu2 / sx_p
        J[:, 6*k+4] = g * ev2 / sy_p
        J[:, 6*k+5] = g * (u*v/sx_p**2 - u*v/sy_p**2)
    return out, J

def volume_mixture(params):
    K = len(params) // 6
    return sum(params[6*k] * 2*np.pi * params[6*k+3] * params[6*k+4] for k in range(K))

class _CachedFit:
    def __init__(self):
        self._p_last = None; self._r_last = None; self._j_last = None
    def update(self, p, x, y, z):
        p = np.asarray(p)
        if self._p_last is None or not np.array_equal(p, self._p_last):
            self._p_last = p.copy()
            fit_v, j_v = gauss_mixture_and_jac(p, x, y)
            self._r_last = fit_v - z; self._j_last = j_v
    def r(self): return self._r_last
    def j(self): return self._j_last

def _make_cached_fit(K_, x_, y_, z_):
    cf = _CachedFit()
    def fun(p): cf.update(p, x_, y_, z_); return cf.r()
    def jac(p): cf.update(p, x_, y_, z_); return cf.j()
    return fun, jac

def _make_cached_fit_pen(K_, x_, y_, z_, dist_px_, grad_c_, grad_r_, pix_mm_, lambda_pen=5.0):
    """Como _make_cached_fit pero añade penalización por centros fuera de la máscara."""
    H_, W_ = dist_px_.shape
    cf = _CachedFit()

    def _eval_pen(p):
        K = len(p) // 6
        pen = np.zeros(K)
        Jp  = np.zeros((K, 6*K))
        for k in range(K):
            c_px = p[6*k+1] / pix_mm_
            r_px = p[6*k+2] / pix_mm_
            ci = int(np.clip(c_px, 0, W_-1))
            ri = int(np.clip(r_px, 0, H_-1))
            d = float(dist_px_[ri, ci]) * pix_mm_   # distancia en mm
            if d > 0:
                pen[k] = lambda_pen * d
                Jp[k, 6*k+1] = lambda_pen * float(grad_c_[ri, ci])
                Jp[k, 6*k+2] = lambda_pen * float(grad_r_[ri, ci])
        return pen, Jp

    def fun(p):
        cf.update(p, x_, y_, z_)
        pen, _ = _eval_pen(p)
        return np.concatenate([cf.r(), pen])

    def jac(p):
        cf.update(p, x_, y_, z_)
        _, Jp = _eval_pen(p)
        return np.vstack([cf.j(), Jp])

    return fun, jac

print('Funciones Multi-Splat definidas.')

Funciones Multi-Splat definidas.


## 6. Multi-Splat: ajuste K=1..51

In [57]:
# ── Parámetros Multi-Splat ───────────────────────────────────────────────────
WIN_DEFECT_MM = 4.0
WIN_DEFECT_PX = int(round(WIN_DEFECT_MM / pix_mm)) | 1
K_MAX         = 15

# Extensión del defecto en pixel-mm (solo píxeles válidos)
rows_def, cols_def = np.where(defect_mask.astype(bool) & valid_mask)
X_MIN_PX = float(cols_def.min()) * pix_mm
X_MAX_PX = float(cols_def.max()) * pix_mm
Y_MIN_PX = float(rows_def.min()) * pix_mm
Y_MAX_PX = float(rows_def.max()) * pix_mm
SPAN_X_PX = X_MAX_PX - X_MIN_PX
SPAN_Y_PX = Y_MAX_PX - Y_MIN_PX
SIG_LONG  = SPAN_X_PX / 2.0
SIG_SHORT = SPAN_Y_PX / 2.0

print(f"Región: X=[{X_MIN_PX:.2f}, {X_MAX_PX:.2f}]  Y=[{Y_MIN_PX:.2f}, {Y_MAX_PX:.2f}] mm")
print(f"SPAN_X={SPAN_X_PX:.2f}  SPAN_Y={SPAN_Y_PX:.2f}  "
      f"SIG_LONG={SIG_LONG:.2f}  SIG_SHORT={SIG_SHORT:.2f}")

def run_multisplat(Z_diff_source, label, lambda_pen=5.0):
    """Ajusta K=1..K_MAX gaussianas a la parte POSITIVA de Z_diff_source.
    Para el lado negativo, llama con -Z_diff.
    lambda_pen: peso de la penalización por centros fuera de la máscara."""
    from scipy.ndimage import distance_transform_edt
    mask_def_b = defect_mask.astype(bool) & valid_mask
    # ── Distancia a la máscara para penalizar centros fuera ──────────────────
    dist_px_ = distance_transform_edt(~mask_def_b).astype(np.float32)
    grad_r_, grad_c_ = np.gradient(dist_px_.astype(np.float64))
    rows_m_, cols_m_ = np.where(mask_def_b)
    x_obs_ = cols_m_.astype(np.float64) * pix_mm
    y_obs_ = rows_m_.astype(np.float64) * pix_mm
    # Solo parte positiva
    z_obs_ = np.clip(Z_diff_source[rows_m_, cols_m_].astype(np.float64), 0.0, None)

    N_obs_ = len(z_obs_)
    N_SUB_ = min(2000, N_obs_)
    rng_   = np.random.default_rng(7)
    idx_   = rng_.choice(N_obs_, N_SUB_, replace=False)
    x_s = x_obs_[idx_]; y_s = y_obs_[idx_]; z_s = z_obs_[idx_]

    A_max_ = max(float(z_obs_.max()) * 3.0, 0.1)
    blo = [0,       X_MIN_PX, Y_MIN_PX, 0.15, 0.15, -np.pi/2]
    bhi = [A_max_,  X_MAX_PX, Y_MAX_PX, SIG_LONG, SIG_SHORT,  np.pi/2]

    coeffs_d_px = sg2d_jacobian_projection_cv2(
        np.where(np.isfinite(Z_diff_source),
                 np.clip(Z_diff_source, 0.0, None), 0.0),
        WIN_DEFECT_PX)
    coeffs_d_ph = np.stack([
        coeffs_d_px[..., 0] / pix_mm**2,
        coeffs_d_px[..., 1] / pix_mm**2,
        coeffs_d_px[..., 2] / pix_mm**2,
        coeffs_d_px[..., 3] / pix_mm,
        coeffs_d_px[..., 4] / pix_mm,
        coeffs_d_px[..., 5]
    ], axis=-1)

    res_all = {}; p_cur = []
    print(f'\n── {label} ──')
    t0 = _time.perf_counter()

    for K in range(1, K_MAX + 1):
        prev_full = gauss_mixture(p_cur, x_obs_, y_obs_) if p_cur else np.zeros(N_obs_)
        res_prev  = np.clip(z_obs_ - prev_full, 0.0, None)
        res_img_  = np.zeros_like(Z_diff_source)
        res_img_[rows_m_, cols_m_] = res_prev
        f_ws = np.where(mask_def_b, res_img_, -np.inf)
        pr, pc = np.unravel_index(np.argmax(f_ws), f_ws.shape)
        mX_ws = float(pc) * pix_mm
        mY_ws = float(pr) * pix_mm

        a0_, b0_, c0_, d0_, e0_, f0_ = coeffs_d_ph[pr, pc, :]
        Hm = np.array([[2*a0_, c0_], [c0_, 2*b0_]])
        try:
            dlt_ = -np.linalg.solve(Hm, [d0_, e0_])
            mX_ws = float(np.clip(mX_ws + dlt_[0], blo[1]+1e-6, bhi[1]-1e-6))
            mY_ws = float(np.clip(mY_ws + dlt_[1], blo[2]+1e-6, bhi[2]-1e-6))
            A_ws_ = float(np.clip(f0_, blo[0]+1e-6, bhi[0]-1e-6))
            Sm = -A_ws_ * np.linalg.inv(Hm) if A_ws_ > 1e-6 else None
            if Sm is not None:
                ev__, evec__ = np.linalg.eigh(Sm)
                ord__ = np.argsort(ev__)[::-1]
                sx_ = float(np.clip(np.sqrt(abs(ev__[ord__[0]])), blo[3]+1e-6, bhi[3]-1e-6))
                sy_ = float(np.clip(np.sqrt(abs(ev__[ord__[1]])), blo[4]+1e-6, bhi[4]-1e-6))
                th_ = float(np.arctan2(evec__[1, ord__[0]], evec__[0, ord__[0]]))
            else:
                sx_, sy_, th_ = SIG_LONG * 0.3, SIG_SHORT * 0.4, 0.0
        except Exception:
            A_ws_ = float(np.clip(res_img_[pr, pc], blo[0]+1e-6, bhi[0]-1e-6))
            sx_, sy_, th_ = SIG_LONG * 0.3, SIG_SHORT * 0.4, 0.0

        p_new_ = [float(np.clip(A_ws_, blo[0]+1e-6, bhi[0]-1e-6)),
                  float(np.clip(mX_ws, blo[1]+1e-6, bhi[1]-1e-6)),
                  float(np.clip(mY_ws, blo[2]+1e-6, bhi[2]-1e-6)),
                  float(np.clip(sx_,   blo[3]+1e-6, bhi[3]-1e-6)),
                  float(np.clip(sy_,   blo[4]+1e-6, bhi[4]-1e-6)),
                  float(np.clip(th_,   blo[5]+1e-6, bhi[5]-1e-6))]
        p0_ = p_cur + p_new_
        fun_, jac_ = _make_cached_fit_pen(K, x_s, y_s, z_s,
                                           dist_px_, grad_c_, grad_r_, pix_mm,
                                           lambda_pen=lambda_pen)
        rK = least_squares(fun_, p0_, jac=jac_,
                           bounds=(blo*K, bhi*K), method='trf',
                           xtol=1e-9, ftol=1e-9, gtol=1e-9, max_nfev=20_000)
        p_cur = list(rK.x)
        V_K_  = volume_mixture(p_cur)
        t_K_  = _time.perf_counter() - t0
        res_all[K] = dict(params=p_cur.copy(), V=V_K_, nfev=rK.nfev, t=t_K_)
        print(f'  K={K}: V={V_K_:.4f} mm³  nfev={rK.nfev}  t={t_K_*1000:.0f}ms')

    return res_all

Z_diff_clean = np.nan_to_num(Z_diff, nan=0.0)

print('=== LADO POSITIVO (protuberancias) ===')
results_pos = run_multisplat(Z_diff_clean,
    f'Polinomial local — lado positivo')

print('\n=== LADO NEGATIVO (hundimientos) ===')
results_neg = run_multisplat(-Z_diff_clean,
    f'Polinomial local — lado negativo')


Región: X=[119.20, 129.23]  Y=[63.60, 75.28] mm
SPAN_X=10.03  SPAN_Y=11.68  SIG_LONG=5.01  SIG_SHORT=5.84
=== LADO POSITIVO (protuberancias) ===

── Polinomial local — lado positivo ──
  K=1: V=9.7546 mm³  nfev=18  t=6ms
  K=2: V=11.5718 mm³  nfev=15  t=15ms
  K=3: V=11.0543 mm³  nfev=14  t=23ms
  K=4: V=11.7005 mm³  nfev=15  t=34ms
  K=5: V=11.1701 mm³  nfev=20  t=71ms
  K=6: V=10.8219 mm³  nfev=18  t=119ms
  K=7: V=10.7651 mm³  nfev=21  t=315ms


  K=8: V=10.7361 mm³  nfev=22  t=709ms
  K=9: V=11.0270 mm³  nfev=21  t=1379ms
  K=10: V=10.9848 mm³  nfev=17  t=2382ms
  K=11: V=11.1288 mm³  nfev=17  t=3790ms
  K=12: V=11.0950 mm³  nfev=37  t=5210ms
  K=13: V=11.2330 mm³  nfev=33  t=5934ms
  K=14: V=11.1066 mm³  nfev=23  t=6149ms
  K=15: V=10.8762 mm³  nfev=29  t=6481ms

=== LADO NEGATIVO (hundimientos) ===

── Polinomial local — lado negativo ──
  K=1: V=7.0631 mm³  nfev=15  t=5ms
  K=2: V=7.0600 mm³  nfev=41  t=21ms
  K=3: V=7.3867 mm³  nfev=22  t=35ms
  K=4: V=7.3067 mm³  nfev=18  t=47ms
  K=5: V=7.0229 mm³  nfev=26  t=85ms
  K=6: V=7.3055 mm³  nfev=21  t=127ms
  K=7: V=7.4384 mm³  nfev=22  t=201ms
  K=8: V=7.3571 mm³  nfev=25  t=286ms
  K=9: V=7.5799 mm³  nfev=21  t=386ms
  K=10: V=7.3235 mm³  nfev=25  t=513ms
  K=11: V=7.3155 mm³  nfev=21  t=659ms
  K=12: V=7.3429 mm³  nfev=35  t=882ms
  K=13: V=7.3748 mm³  nfev=41  t=1134ms
  K=14: V=7.3171 mm³  nfev=41  t=1405ms
  K=15: V=7.4333 mm³  nfev=37  t=1770ms


## 7. Resultados Multi-Splat

In [58]:
%matplotlib tk
from matplotlib.widgets import Slider

def bic_for(results_dict, sign=1):
    """sign=+1 para positivo, -1 para negativo (usamos -Z_diff para neg)."""
    mask_def_b = defect_mask.astype(bool) & valid_mask
    rows_m, cols_m = np.where(mask_def_b)
    xo = cols_m.astype(np.float64) * pix_mm
    yo = rows_m.astype(np.float64) * pix_mm
    zo = np.clip(sign * np.nan_to_num(Z_diff, nan=0.0)[rows_m, cols_m], 0.0, None)
    K_vals_ = sorted(results_dict.keys())
    bics = []
    for k in K_vals_:
        fit = gauss_mixture(results_dict[k]['params'], xo, yo)
        rss = np.sum((zo - fit)**2)
        bics.append(rss + 6*k * np.var(zo))
    K_best_ = K_vals_[int(np.argmin(bics))]
    return bics, K_best_

K_vals = sorted(results_pos.keys())

bic_pos, K_best_pos = bic_for(results_pos, sign=+1)
bic_neg, K_best_neg = bic_for(results_neg, sign=-1)

V_pos = results_pos[K_best_pos]['V']
V_neg = results_neg[K_best_neg]['V']

print(f'POSITIVO: K_best={K_best_pos}  V_pos = +{V_pos:.4f} mm³  (material sobrante)')
print(f'NEGATIVO: K_best={K_best_neg}  V_neg = -{V_neg:.4f} mm³  (material faltante)')
print(f'Desviación neta = {V_pos - V_neg:+.4f} mm³')

print('\n── Tabla positivo ──')
print(f'{"K":>3}  {"V+ (mm³)":>12}  {"BIC":>12}')
print('─' * 32)
for i, k in enumerate(K_vals):
    mark = ' ◄' if k == K_best_pos else ''
    print(f'{k:>3}  {results_pos[k]["V"]:>12.4f}  {bic_pos[i]:>12.4f}{mark}')

print('\n── Tabla negativo ──')
print(f'{"K":>3}  {"V- (mm³)":>12}  {"BIC":>12}')
print('─' * 32)
for i, k in enumerate(K_vals):
    mark = ' ◄' if k == K_best_neg else ''
    print(f'{k:>3}  {results_neg[k]["V"]:>12.4f}  {bic_neg[i]:>12.4f}{mark}')

# ── Figura: V vs K + mapa Z_diff + perfil interactivo ────────────────────────
mask_def_b = defect_mask.astype(bool)

p_best_pos = results_pos[K_best_pos]['params']
p_best_neg = results_neg[K_best_neg]['params']

rows_def_p, cols_def_p = np.where(mask_def_b & valid_mask)
r_center = int(round(rows_def_p.mean()))
row_min  = int(rows_def_p.min())
row_max  = int(rows_def_p.max())

col_slice  = np.arange(c0_bb, c1_bb)
x_slice_mm = col_slice.astype(np.float64) * pix_mm

def get_profile(row):
    y_row = np.full_like(x_slice_mm, float(row) * pix_mm)
    z_sl  = np.nan_to_num(Z_diff, nan=0.0)[row, col_slice]
    fp_sl = gauss_mixture(p_best_pos, x_slice_mm, y_row)
    fn_sl = -gauss_mixture(p_best_neg, x_slice_mm, y_row)
    return z_sl, fp_sl, fn_sl

z_slice, fit_pos_sl, fit_neg_sl = get_profile(r_center)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
plt.subplots_adjust(bottom=0.18)

# Col 1: V vs K (ambos lados)
axes[0].plot(K_vals, [results_pos[k]['V'] for k in K_vals],
             'o-', color='tomato',    lw=2, label='Positivo (+)')
axes[0].plot(K_vals, [-results_neg[k]['V'] for k in K_vals],
             's-', color='steelblue', lw=2, label='Negativo (−)')
axes[0].axvline(K_best_pos, color='tomato',    lw=1.5, ls='--', alpha=0.7)
axes[0].axvline(K_best_neg, color='steelblue', lw=1.5, ls='--', alpha=0.7)
axes[0].axhline(0, color='gray', lw=1, ls=':')
axes[0].set_xlabel('K (nº gaussianas)'); axes[0].set_ylabel('Volumen (mm³)')
axes[0].set_title('Volumen estimado vs K')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Col 2: Z_diff con centros de ambos splats + línea de perfil
roi_r = slice(r0_bb, r1_bb); roi_c = slice(c0_bb, c1_bb)
zdiff_roi = np.nan_to_num(Z_diff, nan=0.0)[roi_r, roi_c]
vmax_z = max(float(np.nanpercentile(np.abs(Z_diff[mask_def_b]), 99)), 0.1)
im2 = axes[1].imshow(zdiff_roi, cmap='RdBu_r', vmin=-vmax_z, vmax=vmax_z, origin='upper')
axes[1].contour(defect_mask[roi_r, roi_c], levels=[0.5], colors='k', linewidths=1.5)
for j in range(len(p_best_pos)//6):
    cx_px = p_best_pos[6*j+1]/pix_mm - c0_bb
    cy_px = p_best_pos[6*j+2]/pix_mm - r0_bb
    sx_px = p_best_pos[6*j+3]/pix_mm
    sy_px = p_best_pos[6*j+4]/pix_mm
    th_deg = np.degrees(p_best_pos[6*j+5])
    axes[1].plot(cx_px, cy_px, '+', color='tomato', ms=12, mew=2.5)
    ell = mpatches.Ellipse((cx_px, cy_px), width=4*sx_px, height=4*sy_px,
                           angle=th_deg, fill=False, edgecolor='tomato',
                           linewidth=1.2, alpha=0.7)
    axes[1].add_patch(ell)
for j in range(len(p_best_neg)//6):
    cx_px = p_best_neg[6*j+1]/pix_mm - c0_bb
    cy_px = p_best_neg[6*j+2]/pix_mm - r0_bb
    sx_px = p_best_neg[6*j+3]/pix_mm
    sy_px = p_best_neg[6*j+4]/pix_mm
    th_deg = np.degrees(p_best_neg[6*j+5])
    axes[1].plot(cx_px, cy_px, 'x', color='dodgerblue', ms=12, mew=2.5)
    ell = mpatches.Ellipse((cx_px, cy_px), width=4*sx_px, height=4*sy_px,
                           angle=th_deg, fill=False, edgecolor='dodgerblue',
                           linewidth=1.2, alpha=0.7)
    axes[1].add_patch(ell)
plt.colorbar(im2, ax=axes[1], label='mm')
axes[1].set_title(f'Z_diff  ✚=splat+  ✕=splat−  (elipses=2σ)')
axes[1].axis('off')
# línea horizontal indicadora de la fila activa
hline_map = axes[1].axhline(r_center - r0_bb, color='lime', lw=1.5, ls='--', alpha=0.9)

# Col 3: perfil interactivo
ax3 = axes[2]
fill_pos = ax3.fill_between(x_slice_mm, 0, z_slice,
                             where=(z_slice > 0), alpha=0.2, color='tomato')
fill_neg = ax3.fill_between(x_slice_mm, 0, z_slice,
                             where=(z_slice < 0), alpha=0.2, color='steelblue')
line_z,   = ax3.plot(x_slice_mm, z_slice,    'k-',  lw=2, label='Z_diff medido')
line_pos, = ax3.plot(x_slice_mm, fit_pos_sl, 'r--', lw=2, label=f'Splat+ K={K_best_pos}')
line_neg, = ax3.plot(x_slice_mm, fit_neg_sl, 'b--', lw=2, label=f'Splat− K={K_best_neg}')
ax3.axhline(0, color='gray', lw=1, ls=':')
ax3.set_xlabel('X (mm)'); ax3.set_ylabel('Z_diff (mm)')
ax3.set_title(f'Perfil horizontal @ fila {r_center}')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

plt.suptitle(f'Multi-Splat ± — {SAMPLE}\n'
             f'V+ = +{V_pos:.3f} mm³  |  V− = −{V_neg:.3f} mm³  |  neto = {V_pos-V_neg:+.3f} mm³',
             fontsize=11)

# ── Slider bajo la tercera columna ──────────────────────────────────────────
# Posición aproximada: misma franja horizontal que axes[2]
ax3_pos = ax3.get_position()
ax_sl = fig.add_axes([ax3_pos.x0, 0.05, ax3_pos.width, 0.04])
slider = Slider(ax_sl, 'Fila', row_min, row_max,
                valinit=r_center, valstep=1, color='limegreen')

def _update(val):
    row = int(slider.val)
    z_sl, fp_sl, fn_sl = get_profile(row)

    # actualizar líneas
    line_z.set_ydata(z_sl)
    line_pos.set_ydata(fp_sl)
    line_neg.set_ydata(fn_sl)

    # rehacer rellenos (fill_between no es fácilmente actualizable)
    global fill_pos, fill_neg
    fill_pos.remove(); fill_neg.remove()
    fill_pos = ax3.fill_between(x_slice_mm, 0, z_sl,
                                where=(z_sl > 0), alpha=0.2, color='tomato')
    fill_neg = ax3.fill_between(x_slice_mm, 0, z_sl,
                                where=(z_sl < 0), alpha=0.2, color='steelblue')

    # reescalar Y automáticamente
    all_vals = np.concatenate([z_sl, fp_sl, fn_sl])
    margin = max(abs(all_vals).max() * 0.15, 0.001)
    ax3.set_ylim(all_vals.min() - margin, all_vals.max() + margin)
    ax3.set_title(f'Perfil horizontal @ fila {row}')

    # mover la línea indicadora en el mapa Z_diff
    hline_map.set_ydata([row - r0_bb, row - r0_bb])

    fig.canvas.draw_idle()

slider.on_changed(_update)

plt.show()


POSITIVO: K_best=7  V_pos = +10.7651 mm³  (material sobrante)
NEGATIVO: K_best=5  V_neg = -7.0229 mm³  (material faltante)
Desviación neta = +3.7422 mm³

── Tabla positivo ──
  K      V+ (mm³)           BIC
────────────────────────────────
  1        9.7546       34.0335
  2       11.5718       17.4119
  3       11.0543       19.4587
  4       11.7005       24.0631
  5       11.1701       19.2210
  6       10.8219       15.3627
  7       10.7651        9.3689 ◄
  8       10.7361       10.5344
  9       11.0270       10.3133
 10       10.9848       10.7088
 11       11.1288       11.2827
 12       11.0950       10.8285
 13       11.2330       10.5815
 14       11.1066       10.4214
 15       10.8762       10.2990

── Tabla negativo ──
  K      V- (mm³)           BIC
────────────────────────────────
  1        7.0631       19.1061
  2        7.0600        6.1471
  3        7.3867        7.3205
  4        7.3067        6.3065
  5        7.0229        4.7549 ◄
  6        7.3055        4.85

## 7b. Métricas geométricas del defecto (splats ajustados)

In [ ]:
# ── Métricas geométricas del defecto según splats ajustados ─────────────────
# Grilla fina sobre la región del defecto
nc_m, nr_m = 300, 150
c_m = np.linspace(float(cols_def.min()), float(cols_def.max()), nc_m)
r_m = np.linspace(float(rows_def.min()), float(rows_def.max()), nr_m)
CM, RM = np.meshgrid(c_m, r_m)
xm_pm = CM * pix_mm
ym_pm = RM * pix_mm

Z_spl_pos = gauss_mixture(p_best_pos, xm_pm.ravel(), ym_pm.ravel()).reshape(nr_m, nc_m)
Z_spl_neg = gauss_mixture(p_best_neg, xm_pm.ravel(), ym_pm.ravel()).reshape(nr_m, nc_m)

# Altura máxima (protuberancia) y profundidad máxima (hundimiento)
h_max = float(Z_spl_pos.max())   # mm, sobrante
h_min = -float(Z_spl_neg.max())  # mm, negativo = hundimiento

# ── Largo y ancho: bounding box orientado (OBB) por PCA ─────────────────────
# sobre los puntos donde la envolvente supera el 5 % del pico
THR_FRAC   = 0.05
Z_envelope = np.maximum(Z_spl_pos, Z_spl_neg)
peak_env   = float(Z_envelope.max())
mask_active = Z_envelope > (THR_FRAC * peak_env)

if mask_active.any():
    r_act, c_act = np.where(mask_active)
    pts_mm = np.column_stack([xm_pm[r_act, c_act],
                               ym_pm[r_act, c_act]])   # (N, 2)

    # PCA → ejes principales
    mu  = pts_mm.mean(axis=0)
    cov = np.cov((pts_mm - mu).T)
    eig_vals, eig_vecs = np.linalg.eigh(cov)
    order = np.argsort(eig_vals)[::-1]   # mayor varianza primero
    eig_vecs = eig_vecs[:, order]

    # Proyectar sobre ejes principales y medir semiejes
    proj = (pts_mm - mu) @ eig_vecs      # (N, 2)
    semieje_largo = float((proj[:, 0].max() - proj[:, 0].min()) / 2)
    semieje_ancho = float((proj[:, 1].max() - proj[:, 1].min()) / 2)
    largo = 2 * semieje_largo
    ancho = 2 * semieje_ancho
    ang_deg = float(np.degrees(np.arctan2(eig_vecs[1, 0], eig_vecs[0, 0])))
else:
    largo = ancho = semieje_largo = semieje_ancho = 0.0
    ang_deg = 0.0
    eig_vecs = np.eye(2)
    mu = np.zeros(2)

print(f"══ Métricas del defecto  (umbral={THR_FRAC*100:.0f}% del pico de la envolvente) ══")
print(f"  Semieje largo (eje 1) : {semieje_largo:.3f} mm   → largo total: {largo:.3f} mm")
print(f"  Semieje corto (eje 2) : {semieje_ancho:.3f} mm   → ancho total: {ancho:.3f} mm")
print(f"  Ángulo eje principal  : {ang_deg:.1f}°  (respecto al eje X global)")
print(f"  Altura máxima (+)     : {h_max:.4f} mm  (protuberancia, K={K_best_pos})")
print(f"  Profundidad máxima (−): {abs(h_min):.4f} mm  (hundimiento,    K={K_best_neg})")
print(f"  Relación largo/ancho  : {largo/ancho:.2f}" if ancho > 0 else "  Relación largo/ancho  : N/A")
print(f"  Volumen sobrante      : +{V_pos:.4f} mm³")
print(f"  Volumen faltante      : -{V_neg:.4f} mm³")
print(f"  Volumen neto          : {V_pos - V_neg:+.4f} mm³")




══ Métricas del defecto  (umbral=5% del pico de la envolvente) ══
  Largo   (dirección X) : 9.526 mm
  Ancho   (dirección Y) : 9.876 mm
  Altura máxima (+)     : 0.6234 mm  (protuberancia, K=7)
  Profundidad máxima (−): 0.6194 mm  (hundimiento,    K=5)
  Relación largo/ancho  : 0.96
  Volumen sobrante      : +10.7651 mm³
  Volumen faltante      : -7.0229 mm³
  Volumen neto          : +3.7422 mm³


## 8. Vista 3D: splats sobre la pieza (todos los K)

In [60]:
%matplotlib tk
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

mask_def_valid = defect_mask.astype(bool) & valid_mask
rows_d, cols_d = np.where(mask_def_valid)

xp_3d = cols_d.astype(np.float64) * pix_mm
yp_3d = rows_d.astype(np.float64) * pix_mm
zp_3d = np.nan_to_num(Z_diff, nan=0.0)[rows_d, cols_d]

# Grilla densa para evaluar splats
nc_grid, nr_grid = 80, 40
c_g = np.linspace(float(cols_d.min()), float(cols_d.max()), nc_grid)
r_g = np.linspace(float(rows_d.min()), float(rows_d.max()), nr_grid)
CG, RG = np.meshgrid(c_g, r_g)
xg_pm = CG * pix_mm
yg_pm = RG * pix_mm

# Rangos para proporciones + exageración Z
x_span = float(xp_3d.max() - xp_3d.min())
y_span = float(yp_3d.max() - yp_3d.min())
vabs   = max(abs(float(zp_3d.min())), abs(float(zp_3d.max())), 0.05)
z_lo_3d = -vabs * 1.1
z_hi_3d =  vabs * 1.1
z_span_3d = z_hi_3d - z_lo_3d
Z_EXAG = max(x_span, y_span) / max(z_span_3d, 0.01) * 0.20
box_aspect = [x_span, y_span, z_span_3d * Z_EXAG]

# ── Figura: solo el mejor K ──────────────────────────────────────────────────
p_pos = results_pos[K_best_pos]['params']
p_neg = results_neg[K_best_neg]['params']

ZG_pos = gauss_mixture(p_pos, xg_pm.ravel(), yg_pm.ravel()).reshape(nr_grid, nc_grid)
ZG_neg = -gauss_mixture(p_neg, xg_pm.ravel(), yg_pm.ravel()).reshape(nr_grid, nc_grid)

zmax_pos = float(ZG_pos.max()) if ZG_pos.max() > 0 else 0.01
zmin_neg = float(ZG_neg.min()) if ZG_neg.min() < 0 else -0.01

fig3d = plt.figure(figsize=(10, 7))
fig3d.suptitle(f'Multi-Splat ± 3D — {SAMPLE}\n'
               f'K+={K_best_pos}  V+=+{V_pos:.3f} mm³  |  K−={K_best_neg}  V−=−{V_neg:.3f} mm³  |  '
               f'neto={V_pos-V_neg:+.3f} mm³\n'
               f'Naranja=protuberancia  Azul=hundimiento  Eje Z ×{Z_EXAG:.0f}',
               fontsize=9)

ax = fig3d.add_subplot(1, 1, 1, projection='3d')

# 1) Nube de puntos
ax.scatter(xp_3d, yp_3d, zp_3d,
           c=zp_3d, cmap='RdBu_r', s=10, alpha=0.8,
           vmin=-vabs, vmax=vabs, depthshade=False)

# 2) Superficie positiva
ax.plot_surface(xg_pm, yg_pm, ZG_pos,
                cmap='YlOrRd', alpha=0.45, shade=False,
                linewidth=0, rstride=1, cstride=1,
                vmin=0, vmax=zmax_pos)

# 3) Superficie negativa
ax.plot_surface(xg_pm, yg_pm, ZG_neg,
                cmap='winter', alpha=0.45, shade=False,
                linewidth=0, rstride=1, cstride=1,
                vmin=zmin_neg, vmax=0)

# 4) Centros de splats
for j in range(len(p_pos)//6):
    ax.scatter([p_pos[6*j+1]], [p_pos[6*j+2]], [p_pos[6*j]],
               color='orangered', s=60, edgecolors='darkred', lw=1.5, zorder=15)
for j in range(len(p_neg)//6):
    ax.scatter([p_neg[6*j+1]], [p_neg[6*j+2]], [-p_neg[6*j]],
               color='cyan', s=60, edgecolors='navy', lw=1.5, zorder=15)

ax.set_xlim(float(xp_3d.min()), float(xp_3d.max()))
ax.set_ylim(float(yp_3d.min()), float(yp_3d.max()))
ax.set_zlim(z_lo_3d, z_hi_3d)
ax.set_box_aspect(box_aspect)
ax.set_xlabel('X (mm)', fontsize=8, labelpad=2)
ax.set_ylabel('Y (mm)', fontsize=8, labelpad=2)
ax.set_zlabel('Z_diff (mm)', fontsize=8, labelpad=2)
ax.tick_params(labelsize=7)
ax.view_init(elev=35, azim=-65)

plt.tight_layout()
plt.show()



In [61]:
%matplotlib tk
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ── Región de interés: bounding box del defecto + margen ────────────────────
pad3d = 30
r0_3d = max(0, int(rows_def_p.min()) - pad3d)
r1_3d = min(H, int(rows_def_p.max()) + pad3d + 1)
c0_3d = max(0, int(cols_def_p.min()) - pad3d)
c1_3d = min(W, int(cols_def_p.max()) + pad3d + 1)

step = 3
rs = np.arange(r0_3d, r1_3d, step)
cs = np.arange(c0_3d, c1_3d, step)
CC3, RR3 = np.meshgrid(cs, rs)

X3  = X[RR3, CC3]
Y3  = Y[RR3, CC3]

valid3   = valid_mask[RR3, CC3]
defect3  = defect_mask[RR3, CC3].astype(bool)

# ── Superficie fantasma: Z real fuera del defecto, Z_nom dentro ──────────────
Zg3 = np.where(defect3, Z_nom[RR3, CC3], Z[RR3, CC3])
z_fill = float(np.nanmean(Zg3[valid3]))
Zn3_f  = np.where(valid3, Zg3, z_fill)

# ── Escalas ──────────────────────────────────────────────────────────────────
x3_span = float(X3[valid3].max() - X3[valid3].min())
y3_span = float(Y3[valid3].max() - Y3[valid3].min())
z3_span = float(np.nanmax(Zn3_f) - np.nanmin(Zn3_f))

# ── Color por píxel ───────────────────────────────────────────────────────────
# gris = dato real fuera del defecto
# naranja = zona del defecto sustituida por Z_nom
# plot_surface con facecolors necesita shape (nr-1, nc-1, 4) — una cara por quad
nr3, nc3 = Zn3_f.shape
vmin_n = float(np.nanpercentile(Zn3_f[valid3 & ~defect3], 1))
vmax_n = float(np.nanpercentile(Zn3_f[valid3 & ~defect3], 99))
norm_z = (Zn3_f - vmin_n) / max(vmax_n - vmin_n, 1e-6)

fc = plt.cm.gray(norm_z)            # base: escala de grises por Z
fc[defect3]  = [1.0, 0.55, 0.0, 0.90]   # defecto → naranja
fc[~valid3]  = [0.5, 0.5,  0.5, 0.40]   # sin dato → gris claro

# Para plot_surface con facecolors hay que dar una cara por quad: (nr-1, nc-1, 4)
fc_faces = fc[:-1, :-1, :]

fig_nom = plt.figure(figsize=(11, 7))
fig_nom.suptitle(f'Superficie fantasma — {SAMPLE}\n'
                 f'Gris = Z real  |  Naranja = defecto sustituido por Z_nom  |  '
                 f'Amarillo = contorno defecto  |  ejes reales en mm',
                 fontsize=9)

ax_n = fig_nom.add_subplot(1, 1, 1, projection='3d')

ax_n.plot_surface(X3, Y3, Zn3_f,
                  facecolors=fc_faces,
                  alpha=0.90, shade=True,
                  linewidth=0, rstride=1, cstride=1)

# ── Contorno de la máscara sobre la superficie nominal ───────────────────────
from scipy.interpolate import griddata
xy_pts_grid = np.column_stack([X3.ravel(), Y3.ravel()])
z_pts_grid  = Zn3_f.ravel()
for pts in polygons:
    pts_mm_x = pts[:, 0].astype(np.float64) * pix_mm
    pts_mm_y = pts[:, 1].astype(np.float64) * pix_mm
    pts_mm_x = np.append(pts_mm_x, pts_mm_x[0])
    pts_mm_y = np.append(pts_mm_y, pts_mm_y[0])
    z_cont = griddata(xy_pts_grid, z_pts_grid,
                      np.column_stack([pts_mm_x, pts_mm_y]),
                      method='linear', fill_value=z_fill)
    ax_n.plot(pts_mm_x, pts_mm_y, z_cont,
              color='yellow', lw=2.5, zorder=20)

ax_n.set_xlim(float(X3[valid3].min()), float(X3[valid3].max()))
ax_n.set_ylim(float(Y3[valid3].min()), float(Y3[valid3].max()))
ax_n.set_zlim(float(np.nanmin(Zn3_f)), float(np.nanmax(Zn3_f)))
ax_n.set_box_aspect([x3_span, y3_span, z3_span])
ax_n.set_xlabel('X (mm)', fontsize=8, labelpad=2)
ax_n.set_ylabel('Y (mm)', fontsize=8, labelpad=2)
ax_n.set_zlabel('Z_nom (mm)', fontsize=8, labelpad=2)
ax_n.tick_params(labelsize=7)
ax_n.view_init(elev=30, azim=-60)

plt.tight_layout()
plt.show()
